# HW03: Variational Autoencoders for Medical Image Generation

**Course**: CSYE 7374 - Deep Learning and Generative AI in Healthcare

---

## Objectives

In this homework, you will apply the Variational Autoencoder techniques covered in class to a **different medical imaging dataset**. You will:

1. Implement a **Convolutional VAE** — encoder, reparameterization trick, and decoder — for RGB pathology images
2. Train the VAE using the **ELBO loss** (reconstruction + KL divergence) with KL annealing
3. Evaluate generation quality via reconstructions, random sampling from the prior, and **latent space interpolation**

---

## Dataset: PathMNIST

**PathMNIST** contains 107,180 RGB colon-cancer pathology slides (28x28 px) labeled across **9 tissue types**:
- 0: adipose
- 1: background
- 2: debris
- 3: lymphocytes
- 4: mucus
- 5: smooth muscle
- 6: normal colon mucosa
- 7: cancer-associated stroma
- 8: colorectal adenocarcinoma epithelium

---

## Instructions

- Complete all cells marked with **`# TODO`**
- Do not modify the provided helper functions unless instructed
- Run all cells in order
- Answer the analysis questions at the end

---

## Grading Rubric

| Task | Points |
|------|--------|
| Data loading and visualization | 10 |
| Data preprocessing (val\_transform) | 5 |
| ConvVAE encoder (`encode` method) | 10 |
| ConvVAE reparameterization trick | 10 |
| ConvVAE decoder (`decode` and `forward` methods) | 10 |
| ELBO loss function (reconstruction + KL) | 10 |
| Optimizer and learning rate scheduler | 5 |
| Training and validation functions | 10 |
| Model training | 5 |
| Latent space interpolation | 5 |
| Analysis questions (4 x 5 points) | 20 |
| **Total** | **100** |

---
## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install -q torch torchvision medmnist matplotlib seaborn scikit-learn tqdm pandas numpy

In [ ]:
import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

import medmnist
from medmnist import INFO

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm

plt.style.use("seaborn-v0_8-whitegrid")
print(f"PyTorch: {torch.__version__}")
print(f"MedMNIST: {medmnist.__version__}")

---
## 2. Configuration

In [ ]:
class Config:
    DATA_FLAG     = "pathmnist"   # PathMNIST colon pathology slides
    DOWNLOAD      = True
    BATCH_SIZE    = 128
    NUM_EPOCHS    = 30
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY  = 1e-5
    LATENT_DIM    = 64
    BETA_MAX      = 1.0           # max KL weight
    IMG_SIZE      = 28
    SEED          = 42
    CHECKPOINT_DIR = "./checkpoints_hw03"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(Config.SEED)
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 3. Load and Explore Data

In [ ]:
info = INFO[Config.DATA_FLAG]
n_channels  = info["n_channels"]
n_classes   = len(info["label"])
class_names = list(info["label"].values())

print(f"Dataset  : {Config.DATA_FLAG.upper()}")
print(f"Channels : {n_channels}  (RGB)")
print(f"Classes  : {n_classes}")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

In [ ]:
DataClass = getattr(medmnist, info["python_class"])

train_dataset_raw = DataClass(split="train", download=Config.DOWNLOAD)

# ============================================================
# TODO: Load the validation and test splits.
# ============================================================
val_dataset_raw  = None  # TODO
test_dataset_raw = None  # TODO
# ============================================================

print(f"Train: {len(train_dataset_raw)} | Val: {len(val_dataset_raw)} | Test: {len(test_dataset_raw)}")

### Visualize Sample Images from Each Class

Complete the function below to display **3 sample images from each of the 9 tissue classes**.

In [ ]:
# ============================================================
# TODO: Visualize sample images from each class.
# ============================================================

def visualize_samples_per_class(dataset, class_names, n_samples=3):
    """Display n_samples images from each class in a grid."""
    n_cls = len(class_names)
    fig, axes = plt.subplots(n_cls, n_samples, figsize=(n_samples * 2, n_cls * 2))

    # TODO: For each class, collect n_samples images and plot them.
    # Raw PathMNIST images are (H, W, C) uint8 numpy arrays.
    # Use axes[class_idx, sample_idx] to place each image.

    pass  # Remove this line after completing TODO

    for ax, name in zip(axes[:, 0], class_names):
        ax.set_ylabel(name, fontsize=7, rotation=0, labelpad=65, va="center")
    plt.suptitle("PathMNIST -- Sample Images per Class", fontsize=13)
    plt.tight_layout(); plt.show()

visualize_samples_per_class(train_dataset_raw, class_names, n_samples=3)
# ============================================================

In [ ]:
# Class distribution (provided)
def get_class_distribution(dataset):
    labels = [int(lbl[0]) for _, lbl in dataset]
    return np.bincount(labels, minlength=n_classes)

train_dist = get_class_distribution(train_dataset_raw)

plt.figure(figsize=(11, 4))
plt.bar(range(n_classes), train_dist, color=sns.color_palette("husl", n_classes))
plt.xlabel("Class Index"); plt.ylabel("Count")
plt.title("PathMNIST -- Training Set Class Distribution")
plt.xticks(range(n_classes),
           [f"{i}\n{n}" for i, n in enumerate(class_names)],
           fontsize=8, rotation=20, ha="right")
plt.tight_layout(); plt.show()

print("Class distribution:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}: {train_dist[i]:,} ({train_dist[i]/train_dist.sum()*100:.1f}%)")

---
## 4. Data Preprocessing

In [ ]:
# Training transform with augmentation (provided)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
])

# ============================================================
# TODO: Define val_transform -- no augmentation, just ToTensor + Normalize.
# ============================================================
val_transform = None  # TODO
# ============================================================

train_dataset = DataClass(split="train", transform=train_transform, download=Config.DOWNLOAD)
val_dataset   = DataClass(split="val",   transform=val_transform,   download=Config.DOWNLOAD)
test_dataset  = DataClass(split="test",  transform=val_transform,   download=Config.DOWNLOAD)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

---
## 5. Model Definition

### ConvVAE Architecture

You will implement a Convolutional VAE for **3-channel 28x28** PathMNIST images.

```
Encoder : Conv(3->32) -> Conv(32->64) -> Flatten -> Linear(3136->256) -> [fc_mu, fc_logvar](256->latent_dim)
Sampling: z = mu + std * eps,   eps ~ N(0, I)   <-- reparameterization trick
Decoder : Linear(latent_dim->3136) -> ConvTranspose(64->32) -> ConvTranspose(32->3) -> Sigmoid
```

The encoder backbone and decoder blocks are **provided**. You will implement the **latent heads** and all four methods: `encode`, `reparameterize`, `decode`, and `forward`.

In [ ]:
class ConvVAE(nn.Module):
    """
    Convolutional Variational Autoencoder for 3-channel 28x28 images.
    Encoder outputs (mu, logvar) for q(z|x) ~ N(mu, diag(exp(logvar))).
    """
    def __init__(self, in_channels=3, latent_dim=64):
        super().__init__()
        self.in_channels = in_channels
        self.latent_dim  = latent_dim

        # -- Encoder backbone (provided -- do not modify) ------------------
        self.enc_backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),  # -> 32x14x14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),           # -> 64x7x7
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.LeakyReLU(0.2),
        )

        # -- Latent heads --------------------------------------------------
        # ============================================================
        # TODO: Define fc_mu and fc_logvar.
        # Both are nn.Linear layers mapping from 256 to latent_dim.
        # ============================================================
        self.fc_mu     = None  # TODO
        self.fc_logvar = None  # TODO
        # ============================================================

        # -- Decoder (provided -- do not modify) ---------------------------
        self.fc_decode = nn.Sequential(
            nn.Linear(latent_dim, 256),     nn.LeakyReLU(0.2),
            nn.Linear(256, 64 * 7 * 7),    nn.LeakyReLU(0.2),
            nn.Unflatten(1, (64, 7, 7)),
        )
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),   # output in [0, 1]
        )

    def encode(self, x):
        """
        Encode x into (mu, logvar).
        Args:    x -- (B, C, H, W)
        Returns: mu, logvar -- each (B, latent_dim)
        """
        # ============================================================
        # TODO: Pass x through self.enc_backbone, then produce
        #       mu with self.fc_mu and logvar with self.fc_logvar.
        #       Return both.
        # ============================================================

        pass  # Remove this line after completing TODO

        # ============================================================

    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + std * eps,  eps ~ N(0, I).
        During eval, return mu directly (deterministic).
        """
        # ============================================================
        # TODO: Implement the reparameterization trick.
        # Hint: std = torch.exp(0.5 * logvar)
        #       Use self.training to switch stochastic vs deterministic.
        # ============================================================

        pass  # Remove this line after completing TODO

        # ============================================================

    def decode(self, z):
        """
        Decode latent vector z back to image space.
        Args:    z -- (B, latent_dim)
        Returns: x_recon -- (B, C, H, W) in [0, 1]
        """
        # ============================================================
        # TODO: Pass z through self.fc_decode then self.dec_conv.
        # ============================================================

        pass  # Remove this line after completing TODO

        # ============================================================

    def forward(self, x):
        """
        Full forward pass.
        Returns: x_recon (B,C,H,W), mu (B,latent_dim), logvar (B,latent_dim)
        """
        # ============================================================
        # TODO: Call encode -> reparameterize -> decode in sequence.
        # ============================================================

        pass  # Remove this line after completing TODO

        # ============================================================

    def sample(self, n, device):
        """Generate n images by sampling z ~ N(0, I). (Provided)"""
        self.eval()
        with torch.no_grad():
            z = torch.randn(n, self.latent_dim).to(device)
            return self.decode(z)

In [ ]:
model = ConvVAE(in_channels=n_channels, latent_dim=Config.LATENT_DIM).to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_p:,}")
print(f"Trainable parameters : {trainable_p:,}")
print(f"Latent dimension     : {Config.LATENT_DIM}")

# Smoke-test forward pass
_x = torch.randn(4, n_channels, Config.IMG_SIZE, Config.IMG_SIZE).to(device)
_r, _mu, _lv = model(_x)
print(f"Input: {_x.shape} | mu: {_mu.shape} | Output: {_r.shape}")
del _x, _r, _mu, _lv

---
## 6. ELBO Loss Function

The VAE minimises the **negative ELBO**:

$$\mathcal{L} = \underbrace{\text{MSE}(\hat{x}, x)}_{\text{reconstruction}} + \beta \cdot \underbrace{(-0.5 \sum (1 + \log\sigma^2 - \mu^2 - \sigma^2))}_{\text{KL divergence}}$$

- **beta** is annealed 0 -> 1 during training to avoid posterior collapse
- Both `x_recon` and `x` should be in **[0, 1]** when computing MSE

In [ ]:
def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    """
    Compute the VAE ELBO loss.

    Args:
        x_recon : (B, C, H, W) in [0, 1]  -- decoder output
        x       : (B, C, H, W) in [0, 1]  -- denormalised target
        mu, logvar : (B, latent_dim)
        beta    : KL weight
    Returns:
        total_loss, recon_loss, kl_loss  (all scalars)
    """
    batch_size = x.size(0)

    # ============================================================
    # TODO: Compute reconstruction loss.
    # Use F.mse_loss(x_recon, x, reduction="sum") / batch_size.
    # ============================================================
    recon_loss = None  # TODO
    # ============================================================

    # ============================================================
    # TODO: Compute KL divergence.
    # KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar)) / batch_size
    # ============================================================
    kl_loss = None  # TODO
    # ============================================================

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

---
## 7. Optimizer and Scheduler

In [ ]:
# ============================================================
# TODO: Create an Adam optimizer and a learning rate scheduler.
# Use Config.LEARNING_RATE and Config.WEIGHT_DECAY for the optimizer.
# Recommended scheduler: ReduceLROnPlateau(factor=0.5, patience=5)
#                     or CosineAnnealingLR(T_max=Config.NUM_EPOCHS)
# ============================================================
optimizer = None  # TODO
scheduler = None  # TODO
# ============================================================

print("Optimizer and scheduler configured.")

---
## 8. Training Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, beta, device):
    """Train for one epoch. Returns avg (total, recon, kl) loss."""
    model.train()
    t_tot = t_rec = t_kl = 0.0

    for images, _ in tqdm(loader, desc="Train", leave=False):
        images    = images.to(device)
        images_01 = (images + 1) / 2   # [-1,1] -> [0,1] to match Sigmoid decoder

        # ============================================================
        # TODO: Complete the training step.
        #   1. Forward pass: x_recon, mu, logvar = model(images)
        #   2. Compute vae_loss() using images_01 as the target
        #   3. optimizer.zero_grad() -> loss.backward()
        #      -> nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        #      -> optimizer.step()
        # ============================================================

        x_recon = None  # TODO
        mu      = None  # TODO
        logvar  = None  # TODO

        loss  = None  # TODO
        recon = None  # TODO
        kl    = None  # TODO

        # TODO: zero_grad -> backward -> clip -> step

        # ============================================================

        t_tot += loss.item(); t_rec += recon.item(); t_kl += kl.item()

    n = len(loader)
    return t_tot / n, t_rec / n, t_kl / n

In [ ]:
def validate(model, loader, beta, device):
    """Validate. Returns avg (total, recon, kl) loss."""
    model.eval()
    v_tot = v_rec = v_kl = 0.0

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Val", leave=False):
            images    = images.to(device)
            images_01 = (images + 1) / 2

            # ============================================================
            # TODO: Forward pass and loss (no backward pass).
            # ============================================================

            x_recon = None  # TODO
            mu      = None  # TODO
            logvar  = None  # TODO

            loss  = None  # TODO
            recon = None  # TODO
            kl    = None  # TODO

            # ============================================================

            v_tot += loss.item(); v_rec += recon.item(); v_kl += kl.item()

    n = len(loader)
    return v_tot / n, v_rec / n, v_kl / n

In [ ]:
# Outer training loop (provided -- do not modify)
def train_model(model, train_loader, val_loader, optimizer, scheduler,
                num_epochs, device, checkpoint_dir):
    history = {k: [] for k in ["train_total", "train_recon", "train_kl",
                                "val_total",   "val_recon",   "val_kl"]}
    best_val  = float("inf")
    best_path = os.path.join(checkpoint_dir, "best_vae.pth")

    for epoch in range(num_epochs):
        beta = min(Config.BETA_MAX,
                   (epoch + 1) / (num_epochs * 0.5) * Config.BETA_MAX)

        tr_tot, tr_rec, tr_kl = train_one_epoch(model, train_loader, optimizer, beta, device)
        vl_tot, vl_rec, vl_kl = validate(model, val_loader, beta, device)

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(vl_tot)
        else:
            scheduler.step()

        for k, v in zip(history, [tr_tot, tr_rec, tr_kl, vl_tot, vl_rec, vl_kl]):
            history[k].append(v)

        if vl_tot < best_val:
            best_val = vl_tot
            torch.save(model.state_dict(), best_path)

        if (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch+1:3d}/{num_epochs} [beta={beta:.2f}] | "
                  f"Train {tr_tot:.3f} (R={tr_rec:.3f} KL={tr_kl:.3f}) | "
                  f"Val {vl_tot:.3f} | LR={lr:.2e}")

    print(f"\nBest val loss: {best_val:.4f}")
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
    return history

---
## 9. Train the Model

Training 30 epochs on CPU takes ~20-40 min. On a GPU it is typically under 5 min.

In [ ]:
# ============================================================
# TODO: Call train_model() with the correct arguments and store
#       the returned history in a variable called `history`.
# ============================================================

history = None  # TODO

# ============================================================

print("Training complete!")

---
## 10. Learning Curves

In [ ]:
def plot_training_curves(history):
    """Plot total, recon, and KL loss. (Provided)"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    pairs = [
        ("train_total", "val_total", "Total ELBO Loss"),
        ("train_recon", "val_recon", "Reconstruction Loss"),
        ("train_kl",   "val_kl",   "KL Divergence"),
    ]
    for ax, (tk, vk, title) in zip(axes, pairs):
        ax.plot(history[tk], label="Train")
        ax.plot(history[vk], label="Validation")
        ax.set_title(title); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("VAE Training History -- PathMNIST", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

# ============================================================
# TODO: Call plot_training_curves with your history dictionary.
# ============================================================

# plot_training_curves(history)

# ============================================================

---
## 11. Reconstruction Quality

In [ ]:
def visualize_reconstructions(model, loader, device, n=8):
    """Show original vs reconstructed images. (Provided)"""
    model.eval()
    images, _ = next(iter(loader))
    images_01 = ((images[:n] + 1) / 2).clamp(0, 1)

    with torch.no_grad():
        recon, _, _ = model(images[:n].to(device))
    recon = recon.cpu().clamp(0, 1)

    fig, axes = plt.subplots(2, n, figsize=(n * 2, 4))
    for i in range(n):
        axes[0, i].imshow(images_01[i].permute(1, 2, 0)); axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].permute(1, 2, 0));     axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original",      fontsize=10)
    axes[1, 0].set_ylabel("Reconstructed", fontsize=10)
    plt.suptitle("VAE Reconstruction -- PathMNIST", fontsize=13)
    plt.tight_layout(); plt.show()

    mse  = F.mse_loss(recon, images_01).item()
    psnr = 10 * np.log10(1.0 / (mse + 1e-8))
    print(f"Reconstruction MSE : {mse:.5f}")
    print(f"Reconstruction PSNR: {psnr:.2f} dB")

# ============================================================
# TODO: Call visualize_reconstructions using the test_loader.
# ============================================================

# visualize_reconstructions(model, test_loader, device)

# ============================================================

---
## 12. Generate New Samples

A well-trained VAE should produce coherent pathology images when sampling z ~ N(0, I).

In [ ]:
def show_generated_samples(model, n, device):
    """Generate and display n images from the prior. (Provided)"""
    model.eval()
    samples = model.sample(n, device).cpu().clamp(0, 1)
    cols = min(n, 8); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = np.array(axes).flatten()
    for i in range(n):
        axes[i].imshow(samples[i].permute(1, 2, 0)); axes[i].axis("off")
    for ax in axes[n:]: ax.axis("off")
    plt.suptitle(f"VAE: {n} Samples Generated from N(0,I)", fontsize=13)
    plt.tight_layout(); plt.show()

# ============================================================
# TODO: Call show_generated_samples to display 16 generated images.
# ============================================================

# show_generated_samples(model, 16, device)

# ============================================================

---
## 13. Latent Space Exploration

### PCA of Encoded Test Images

In [ ]:
# Collect encoded mu for all test images (provided)
model.eval()
all_mu, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        mu, _ = model.encode(images.to(device))
        all_mu.append(mu.cpu())
        all_labels.append(labels.squeeze())

all_mu     = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

pca_2d  = PCA(n_components=2).fit_transform(all_mu)
z_range = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for c in range(n_classes):
    mask = all_labels == c
    axes[0].scatter(pca_2d[mask, 0], pca_2d[mask, 1],
                   label=class_names[c], alpha=0.35, s=5)
axes[0].set_title("VAE Latent Space (PCA) -- PathMNIST")
axes[0].legend(markerscale=2, loc="best", fontsize=7, title="Tissue")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")

axes[1].hist(all_mu.flatten(), bins=100, density=True, alpha=0.7, label="VAE mu")
axes[1].plot(z_range, np.exp(-0.5 * z_range**2) / np.sqrt(2 * np.pi),
             "r--", lw=2, label="N(0,1)")
axes[1].set_title("Latent Distribution vs Prior")
axes[1].set_xlabel("Value"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Latent mean: {all_mu.mean():.3f}  (target ~0)")
print(f"Latent std : {all_mu.std():.3f}  (target ~1)")

### Latent Space Interpolation

Interpolate smoothly between two test images in latent space.
A well-regularised VAE produces a gradual, semantically meaningful transition.

In [ ]:
# ============================================================
# TODO: Implement latent-space interpolation.
#
# Steps:
#   1. Encode img_a and img_b with model.encode() to get mu_a, mu_b.
#   2. Create 10 interpolated latent vectors:
#      z_t = (1 - alpha) * mu_a + alpha * mu_b  for alpha in linspace(0,1,10)
#   3. Decode each z_t. Store results in interp_imgs (shape: 10, C, H, W).
# ============================================================

model.eval()

test_batch, _ = next(iter(test_loader))
img_a = test_batch[0:1].to(device)   # (1, 3, 28, 28)
img_b = test_batch[1:2].to(device)   # (1, 3, 28, 28)

with torch.no_grad():
    interp_imgs = None  # TODO: shape (10, 3, 28, 28)

# ============================================================

# Plotting scaffold (provided -- runs once interp_imgs is filled in)
if interp_imgs is not None:
    alphas = torch.linspace(0, 1, 10)
    a_show = ((img_a[0].cpu() + 1) / 2).clamp(0, 1)
    b_show = ((img_b[0].cpu() + 1) / 2).clamp(0, 1)

    fig, axes = plt.subplots(1, 12, figsize=(22, 2.5))
    axes[0].imshow(a_show.permute(1, 2, 0)); axes[0].set_title("A"); axes[0].axis("off")
    for i, img in enumerate(interp_imgs.cpu().clamp(0, 1)):
        axes[i + 1].imshow(img.permute(1, 2, 0))
        axes[i + 1].set_title(f"{alphas[i]:.1f}"); axes[i + 1].axis("off")
    axes[11].imshow(b_show.permute(1, 2, 0)); axes[11].set_title("B"); axes[11].axis("off")
    plt.suptitle("Latent Interpolation: A -> B (PathMNIST VAE)", y=1.08)
    plt.tight_layout(); plt.show()

---
## 14. Analysis Questions (20 points)

Answer the following questions based on your results. Write your answers in the markdown cells below each question.

### Question 1 (5 points)

Examine your **learning curves**. At what epoch does the KL divergence begin to rise significantly? How does this relate to the KL annealing schedule (beta ramps from 0 to 1 over the first 15 epochs)? What would likely happen to reconstruction quality if beta were fixed at 1.0 from epoch 1?

**Your Answer:**

*[Write your answer here]*

### Question 2 (5 points)

In the lecture, the plain autoencoder's latent space was **unregularised** -- random sampling from N(0, I) produced incoherent images. Inspect your **latent distribution histogram**: how closely does your VAE posterior match N(0, 1)? Do the generated samples (Section 12) look like plausible pathology tiles? Explain the role of the KL term in enabling coherent generation.

**Your Answer:**

*[Write your answer here]*

### Question 3 (5 points)

Look at the **PCA latent space plot**. Identify two tissue types that appear difficult to separate in latent space. Provide one visual reason (based on what the tissue images look like) and one model-level reason (based on the VAE architecture or training objective) why these classes might be conflated.

**Your Answer:**

*[Write your answer here]*

### Question 4 (5 points)

The lecture used a deeper **U-Net-style VAE** (4 encoder stages, 128x128 images) for chest X-rays, while this homework uses a 2-stage ConvVAE on 28x28 PathMNIST patches. Discuss the trade-offs between these two designs in terms of reconstruction quality, generation diversity, and training cost. If you were designing a VAE for 512x512 whole-slide histology images, what two architectural changes would you make and why?

**Your Answer:**

*[Write your answer here]*

---
## Submission Instructions

1. Run all cells and ensure there are no errors
2. Rename this notebook as: **`FirstName_LastName_HW03.ipynb`**
3. Download and submit to the course portal by the deadline

**Example filename:** `John_Smith_HW03.ipynb`

**Make sure your notebook shows:**
- All TODOs completed
- Training output visible (at least every-5-epoch log lines)
- Learning curves, reconstruction grid, generated samples, and interpolation rendered
- Analysis questions answered